# Cerebro Unsloth Fine-Tuning

Fast LoRA/QLoRA - 2x faster, 70% less VRAM

## Setup: Runtime > Change runtime type > GPU (T4)


In [ ]:
!pip install unsloth peft trl accelerate --quiet


In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="cerebro-tiny", max_seq_length=2048,
    dtype=None, load_in_4bit=True)
print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16, lora_dropout=0.05, bias="none",
    use_gradient_checkpointing="unsloth", random_state=3407)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from datasets import load_dataset
dataset = load_dataset("json", data_files="/content/drive/MyDrive/train.jsonl", split="train")
print(f"Dataset: {len(dataset)} samples")


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=dataset, dataset_text_field="text",
    max_seq_length=2048, dataset_num_proc=2, packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=5, max_steps=200, learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="linear", seed=3407,
        output_dir="outputs", report_to="none"))
trainer.train()


In [ ]:
model.save_pretrained("cerebro_lora")
tokenizer.save_pretrained("cerebro_lora")
import shutil
from google.colab import files
shutil.make_archive("/content/cerebro_lora", "zip", "cerebro_lora")
files.download("/content/cerebro_lora.zip")
